# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`

This notebook showcases how to load and explore the FAIR^2 dataset using the `mlcroissant` library with a Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets, fields, and their `@id`s.

In [ ]:
# Get list of record sets by @id
record_sets = []
for rs in metadata.recordSet:
    record_sets.append(rs['@id'])

print('Available record sets and fields:')
record_set_to_fields = {}
for rs in metadata.recordSet:
    rs_id = rs['@id']
    print(f"- RecordSet @id: {rs_id}")
    fields = [f['@id'] for f in rs['field']]
    record_set_to_fields[rs_id] = fields
    print(f"  Fields @id: {fields}")

## 3. Data Extraction
Load each record set into a DataFrame for analysis, using their `@id` values.

In [ ]:
# Extract data from all available record sets
dataframes = {}
for rs_id in record_sets:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from RecordSet @id: {rs_id}")
    except Exception as e:
        print(f"Skipped RecordSet {rs_id}: {e}")

# List the columns (field @id) of the main record set
if len(dataframes):
    main_rs_id = record_sets[0]
    print(f"Fields in {main_rs_id}:\n", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No dataframes were loaded. Please check the recordSets and the dataset integrity.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing such as filtering, normalizing, and grouping on numeric and categorical fields using field `@id`s.

In [ ]:
# For demonstration, pick the first record set and its numeric fields (by inspecting column names)
import numpy as np

main_df = dataframes[main_rs_id].copy()
numeric_field_id = None
# Try to infer numeric fields by dtype or field naming:
for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Try typical field names if inference fails (manual fallback)
    for nf in ['cr:Age', 'cr:Interval_Months', 'cr:Comorbidity_Count', 'cr:Total_Interval']:
        if nf in main_df.columns:
            numeric_field_id = nf
            break

if numeric_field_id is None:
    raise Exception("No suitable numeric field found by @id in record set.")
print(f"Using numeric field @id for EDA: {numeric_field_id}")

# Filter records with value > threshold
threshold = main_df[numeric_field_id].mean()
filtered_df = main_df.loc[main_df[numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold}:")
display(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / (filtered_df[numeric_field_id].std() + 1e-9)

print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field (@id). Try typical ones like 'cr:Sex', 'cr:Location', etc.
group_field_id = None
for group_cand in ['cr:Sex', 'cr:Location', 'cr:MSI_Status']:
    if group_cand in main_df.columns:
        group_field_id = group_cand
        break
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())
else:
    print('No suitable group field found for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7,4))
sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=15)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# Boxplot by group if available
if group_field_id and group_field_id in main_df.columns:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()
else:
    print("No suitable group categorical field for boxplot.")

## 6. Conclusion
Through the `mlcroissant` library, we loaded and explored the FAIR^2 dataset using Croissant's interoperable data model. We listed record sets and fields by their `@id`, extracted them into DataFrames, applied filtering and normalization, and visualized core data distributions.

This approach paves the way for reproducible and FAIR-compliant data science workflows using the Croissant metadata standard.